In [1]:
!pip install -q sentence-transformers chromadb ragas datasets
!pip install -q transformers accelerate bitsandbytes
!pip install -q pandas numpy tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [2]:
import os
import json
import ast
import random
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU check
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/CS455")
DATA_DIR = PROJECT_DIR
CACHE_DIR = PROJECT_DIR / "hw2_cache"
CACHE_DIR.mkdir(exist_ok=True)

CHROMA_DIR_SMALL = str(CACHE_DIR / "chroma_bge_small")
CHROMA_DIR_BASE = str(CACHE_DIR / "chroma_bge_base")

print(f"\nProject dir: {PROJECT_DIR}")
print(f"Files in project dir:")
for f in sorted(PROJECT_DIR.iterdir()):
    print(f"  {f.name}")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB
Mounted at /content/drive

Project dir: /content/drive/MyDrive/Colab Notebooks/CS455
Files in project dir:
  CS455_HW2_MovieRAG.ipynb
  credits.csv
  hw2_cache
  movies_metadata.csv


In [3]:
print("Loading CSVs...")
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)

credits = pd.read_csv(
    DATA_DIR / "credits.csv",
    engine='python',
    on_bad_lines='skip'
)
print(f"Raw movies: {len(movies):,}")
print(f"Raw credits: {len(credits):,}")

# Clean movies
# 1. The 'id' column has a few rows with garbage strings
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# 2. Drop rows with missing title
movies = movies.dropna(subset=['title'])

# 3. Drop rows with missing/empty overview
movies = movies[movies['overview'].notna()]
movies = movies[movies['overview'].str.strip().str.len() > 0]

# 4. Drop duplicate titles, keeping the longer overview
movies['_overview_len'] = movies['overview'].str.len()
movies = movies.sort_values('_overview_len', ascending=False)
movies = movies.drop_duplicates(subset=['title'], keep='first')
movies = movies.drop(columns=['_overview_len'])

print(f"After cleaning movies: {len(movies):,}")

# Clean credits
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id'])
credits['id'] = credits['id'].astype(int)

# Merge
df = movies.merge(credits, on='id', how='inner')
print(f"After merge: {len(df):,}")

# Parse stringified JSON columns
def safe_parse(x):
    """The CSV stores lists/dicts as Python repr strings."""
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df['genres_parsed'] = df['genres'].apply(safe_parse)
df['cast_parsed'] = df['cast'].apply(safe_parse)
df['crew_parsed'] = df['crew'].apply(safe_parse)

# Extract clean fields
def extract_genres(parsed):
    return [g['name'] for g in parsed if isinstance(g, dict) and 'name' in g]

def extract_director(crew):
    for person in crew:
        if isinstance(person, dict) and person.get('job') == 'Director':
            return person.get('name', '')
    return ''

def extract_top_cast(cast, n=5):
    return [p['name'] for p in cast[:n] if isinstance(p, dict) and 'name' in p]

def extract_year(date_str):
    if pd.isna(date_str):
        return None
    try:
        return int(str(date_str)[:4])
    except (ValueError, TypeError):
        return None

df['genre_names'] = df['genres_parsed'].apply(extract_genres)
df['director'] = df['crew_parsed'].apply(extract_director)
df['top_cast'] = df['cast_parsed'].apply(extract_top_cast)
df['release_year'] = df['release_date'].apply(extract_year)

# Final usable columns
df = df[['id', 'title', 'overview', 'release_year', 'genre_names', 'director', 'top_cast']].copy()
df = df.reset_index(drop=True)

print(f"\n✅ Final corpus: {len(df):,} movies")
print(f"\nSample row:")
print(df.iloc[0].to_dict())

Loading CSVs...
Raw movies: 45,466
Raw credits: 37,257
After cleaning movies: 41,367
After merge: 33,938

✅ Final corpus: 33,938 movies

Sample row:
{'id': 267048, 'title': 'The Pyramid', 'overview': "The film is set in the middle of winter in Ystad and the main character is the middle-aged policeman Kurt Wallander, who this time investigating a drug tangle. It starts with his own goddaughter Eva found dead after taking an overdose. It turns out that heroin is unusually strong and Kurt are now beginning their search for those behind the drugs. But he has a big problem. His boss will not let him look for the person who he suspects has sold the new heroin , but she wants them to look for another drug dealer named Yngve Holm. But Kurt defy the boss's orders and go again and again to Malmo to try to unravel the case. Eventually he gets a hold of the person he suspected. He'll take some photos of him and shows them for their guddotters friend Emma who were drug dealers. She takes Kurt to a 

In [4]:
def build_document_text(row):
    """
    Format:
    - Title appears twice (with year, then alone) to weight title tokens more heavily.
    - Helps with sequel-confusion: 'Toy Story (1995)' embeds differently from 'Toy Story 3 (2010)'.
    """
    title = row['title']
    year = row['release_year']
    genres = ', '.join(row['genre_names']) if row['genre_names'] else 'Unknown'
    director = row['director'] if row['director'] else 'Unknown'
    cast = ', '.join(row['top_cast']) if row['top_cast'] else 'Unknown'
    overview = row['overview']
    year_str = f"({int(year)})" if pd.notna(year) else ""

    return (
        f"{title} {year_str}. {title}. "
        f"Genres: {genres}. "
        f"Directed by {director}. "
        f"Starring {cast}. "
        f"Plot: {overview}"
    )

df['document_text'] = df.apply(build_document_text, axis=1)

# Corpus statistics for Analysis Report
overview_lens = df['overview'].str.split().str.len()  # word-count proxy for tokens
print(f"Final corpus size: {len(df):,} movies")
print(f"\nOverview length (in words, ~tokens):")
print(f"  Mean:   {overview_lens.mean():.1f}")
print(f"  Median: {overview_lens.median():.1f}")
print(f"  Max:    {overview_lens.max()}")
print(f"  Min:    {overview_lens.min()}")

# How many movies are missing year / director? Useful for Analysis Report.
print(f"\nMissing metadata:")
print(f"  No release year: {df['release_year'].isna().sum():,}")
print(f"  No director:     {(df['director'] == '').sum():,}")
print(f"  No cast:         {(df['top_cast'].str.len() == 0).sum():,}")

print(f"\ 3 example document_text strings\n")
for i in [0, len(df)//2, len(df)-1]:
    print(f"[{i}] {df.iloc[i]['document_text'][:300]}...")
    print()

<>:40: SyntaxWarning: invalid escape sequence '\ '
<>:40: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_3992/1692771615.py:40: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ 3 example document_text strings\n")


Final corpus size: 33,938 movies

Overview length (in words, ~tokens):
  Mean:   56.5
  Median: 50.0
  Max:    187
  Min:    1

Missing metadata:
  No release year: 39
  No director:     537
  No cast:         1,570
\ 3 example document_text strings

[0] The Pyramid (2007). The Pyramid. Genres: Thriller, Action, Crime. Directed by Daniel Lind Lagerlöf. Starring Rolf Lassgård, Marie Richardson, Kerstin Andersson, Lars Melin, Gunilla Abrahamsson. Plot: The film is set in the middle of winter in Ystad and the main character is the middle-aged policeman...

[16969] Gutterballs (2008). Gutterballs. Genres: Horror. Directed by Ryan Nicholson. Starring Alastair Gamble, Mihola Terzic, Nathan Witte, Wade Gibb, Candice Lewald. Plot: A brutally sadistic rape leads to a series of bizarre gory murders during a midnight disco bowl-a-rama at a popular bowling alley. One ...

[33937] Wojaczek (1999). Wojaczek. Genres: Drama. Directed by Lech Majewski. Starring Unknown. Plot: x...



In [5]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_SMALL = "BAAI/bge-small-en-v1.5"
EMBED_MODEL_BASE = "BAAI/bge-base-en-v1.5"  # used in Ablation A later

print(f"Loading {EMBED_MODEL_SMALL}...")
embedder = SentenceTransformer(EMBED_MODEL_SMALL, device='cuda')
print(f"✅ Loaded. Embedding dimension: {embedder.get_sentence_embedding_dimension()}")
print(f"   Max seq length: {embedder.max_seq_length}")

Loading BAAI/bge-small-en-v1.5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Loaded. Embedding dimension: 384
   Max seq length: 512


/tmp/ipykernel_3992/2871557753.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Loaded. Embedding dimension: {embedder.get_sentence_embedding_dimension()}")


In [6]:
documents = df['document_text'].tolist()
print(f"Encoding {len(documents):,} documents...")

start = time.perf_counter()
doc_embeddings = embedder.encode(
    documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
elapsed = time.perf_counter() - start

throughput = len(documents) / elapsed
mem_gb = doc_embeddings.nbytes / 1e9

print(f"\n✅ Encoded {len(documents):,} docs in {elapsed:.1f}s")
print(f"   Throughput: {throughput:.0f} docs/sec")
print(f"   Embedding shape: {doc_embeddings.shape}")
print(f"   Memory footprint: {mem_gb*1000:.1f} MB")

Encoding 33,938 documents...


Batches:   0%|          | 0/266 [00:00<?, ?it/s]


✅ Encoded 33,938 docs in 27.0s
   Throughput: 1255 docs/sec
   Embedding shape: (33938, 384)
   Memory footprint: 52.1 MB


In [8]:
import chromadb

print(f"Persisting Chroma index to: {CHROMA_DIR_SMALL}")

before = len(df)
keep_mask = ~df['id'].duplicated(keep='first')
df = df.loc[keep_mask].reset_index(drop=True)
doc_embeddings = doc_embeddings[keep_mask.values]
documents = df['document_text'].tolist()
print(f"Deduped: {before:,} → {len(df):,} rows ({before - len(df)} duplicates removed)")

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR_SMALL)
COLLECTION_NAME = "movies_bge_small"

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("  Deleted existing collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print("Preparing metadata...")
ids = [str(mid) for mid in df['id'].tolist()]
metadatas = [
    {
        "title": row['title'],
        "year": int(row['release_year']) if pd.notna(row['release_year']) else -1,
        "genres": ', '.join(row['genre_names']) if row['genre_names'] else '',
        "director": row['director'] if row['director'] else '',
        "cast": ', '.join(row['top_cast']) if row['top_cast'] else '',
    }
    for _, row in df.iterrows()
]

assert len(set(ids)) == len(ids), "Still have duplicate IDs!"
assert len(ids) == len(documents) == len(metadatas) == len(doc_embeddings)

BATCH = 5000
print(f"Inserting {len(ids):,} documents in batches of {BATCH}...")
start = time.perf_counter()
for i in tqdm(range(0, len(ids), BATCH)):
    end = min(i + BATCH, len(ids))
    collection.add(
        ids=ids[i:end],
        embeddings=doc_embeddings[i:end].tolist(),
        documents=documents[i:end],
        metadatas=metadatas[i:end],
    )
elapsed = time.perf_counter() - start

print(f"\n✅ Indexed {collection.count():,} documents in {elapsed:.1f}s")
print(f"   Index location: {CHROMA_DIR_SMALL}")

Persisting Chroma index to: /content/drive/MyDrive/Colab Notebooks/CS455/hw2_cache/chroma_bge_small
Deduped: 33,938 → 33,907 rows (31 duplicates removed)
  Deleted existing collection.
Preparing metadata...
Inserting 33,907 documents in batches of 5000...


  0%|          | 0/7 [00:00<?, ?it/s]


✅ Indexed 33,907 documents in 215.0s
   Index location: /content/drive/MyDrive/Colab Notebooks/CS455/hw2_cache/chroma_bge_small


In [9]:
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def encode_query(query: str) -> np.ndarray:
    """Encode a single query with BGE's recommended prefix + normalization."""
    return embedder.encode(
        [BGE_QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]

test_query = "Who directed Inception?"
q_emb = encode_query(test_query)

results = collection.query(
    query_embeddings=[q_emb.tolist()],
    n_results=5,
)

print(f"Query: {test_query}\n")
print(f"{'Rank':<6}{'Score':<10}{'Title':<40}{'Director':<25}")
print("-" * 80)
for rank, (doc_id, dist, meta) in enumerate(zip(
    results['ids'][0],
    results['distances'][0],
    results['metadatas'][0],
), start=1):
    score = 1 - dist
    title = meta['title'][:38]
    director = meta['director'][:23] if meta['director'] else 'N/A'
    print(f"{rank:<6}{score:<10.4f}{title:<40}{director:<25}")

Query: Who directed Inception?

Rank  Score     Title                                   Director                 
--------------------------------------------------------------------------------
1     0.7821    Inception                               Christopher Nolan        
2     0.6547    The American Dreamer                    L.M. Kit Carson          
3     0.6322    Dreamscape                              Joseph Ruben             
4     0.6247    Dreams on Spec                          Daniel Snyder            
5     0.6227    Hollywood between Paranoia and Sci-Fi.  Julia Kuperberg          


In [10]:
from sentence_transformers import CrossEncoder

CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print(f"Loading cross-encoder: {CROSS_ENCODER_NAME}")
reranker = CrossEncoder(CROSS_ENCODER_NAME, device='cuda', max_length=512)
print("✅ Cross-encoder loaded")

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Cross-encoder loaded


In [11]:
def retrieve(query: str, k: int = 20, target_collection=None):
    """
    Bi-encoder retrieval. Returns list of dicts with title, director, score, document_text.

    Args:
        query: natural language query
        k: number of candidates to retrieve
        target_collection: which Chroma collection to query (defaults to global `collection`)
    """
    if target_collection is None:
        target_collection = collection

    q_emb = encode_query(query)
    results = target_collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=k,
    )

    candidates = []
    for doc_id, dist, meta, doc in zip(
        results['ids'][0],
        results['distances'][0],
        results['metadatas'][0],
        results['documents'][0],
    ):
        candidates.append({
            'id': doc_id,
            'title': meta['title'],
            'year': meta.get('year', -1),
            'director': meta.get('director', ''),
            'genres': meta.get('genres', ''),
            'cast': meta.get('cast', ''),
            'document_text': doc,
            'bi_score': 1 - dist,  # convert cosine distance to similarity
        })
    return candidates


def rerank(query: str, candidates: list, n: int = 5):
    """
    Cross-encoder re-ranks candidates. Returns top-n with new 'rerank_score' field.
    """
    if not candidates:
        return []

    pairs = [[query, c['document_text']] for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)

    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)

    candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
    return candidates[:n]

In [12]:
DEMO_QUERIES = [
    "Who directed Inception?",
    "A movie about a hacker who discovers reality is a simulation",
    "Pixar animated film about toys that come to life",
]

def show_comparison(query):
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print('='*80)

    candidates = retrieve(query, k=20)

    print(f"\n--- Top-5 from BI-ENCODER ALONE ---")
    print(f"{'Rank':<6}{'Bi-score':<12}{'Title':<45}{'Year':<6}")
    for i, c in enumerate(candidates[:5], 1):
        print(f"{i:<6}{c['bi_score']:<12.4f}{c['title'][:43]:<45}{c['year']:<6}")

    reranked = rerank(query, candidates, n=5)

    print(f"\n--- Top-5 AFTER RE-RANKING ---")
    print(f"{'Rank':<6}{'Rerank':<12}{'Bi-score':<12}{'Title':<45}{'Year':<6}")
    for i, c in enumerate(reranked, 1):
        print(f"{i:<6}{c['rerank_score']:<12.4f}{c['bi_score']:<12.4f}{c['title'][:43]:<45}{c['year']:<6}")

for q in DEMO_QUERIES:
    show_comparison(q)


Query: Who directed Inception?

--- Top-5 from BI-ENCODER ALONE ---
Rank  Bi-score    Title                                        Year  
1     0.7821      Inception                                    2010  
2     0.6547      The American Dreamer                         1971  
3     0.6322      Dreamscape                                   1984  
4     0.6247      Dreams on Spec                               2007  
5     0.6227      Hollywood between Paranoia and Sci-Fi. The   2011  

--- Top-5 AFTER RE-RANKING ---
Rank  Rerank      Bi-score    Title                                        Year  
1     9.3894      0.7821      Inception                                    2010  
2     -1.8946     0.6071      Transcendence                                2014  
3     -2.9013     0.6191      Ambition                                     2014  
4     -3.0746     0.6156      TerrorVision                                 1986  
5     -3.1643     0.6227      Hollywood between Paranoia and Sci-Fi. 

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {LLM_NAME}...")
print("(First run downloads ~6 GB)")

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="cuda",
)
llm.eval()

print(f"✅ LLM loaded")
print(f"   Parameters: {sum(p.numel() for p in llm.parameters())/1e9:.2f}B")
print(f"   GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")

Loading Qwen/Qwen2.5-3B-Instruct...
(First run downloads ~6 GB)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM loaded
   Parameters: 3.09B
   GPU memory: 6.4 GB allocated


In [15]:
PERMISSIVE_PROMPT = """You are a helpful movie expert. Use the context below to answer the user's question. Mention the movie titles you used."""

STRICT_PROMPT = """You are a movie expert that answers ONLY from the provided context.

RULES:
1. Use ONLY information explicitly stated in the context. Do not use outside knowledge.
2. If the context does not contain enough information to answer, respond EXACTLY: "I don't have information about that in the provided context."
3. Always cite the movie title(s) from the context that you used, in the format: [Title (Year)].
4. If the question asks about a movie not in the context, refuse with the exact phrase above.

EXAMPLE 1 (answer is in context):
Context: Inception (2010). Directed by Christopher Nolan. Plot: A thief steals dreams...
Question: Who directed Inception?
Answer: Christopher Nolan directed Inception (2010). [Inception (2010)]

EXAMPLE 2 (answer is NOT in context):
Context: The Godfather (1972). Directed by Francis Ford Coppola.
Question: Who directed Oppenheimer?
Answer: I don't have information about that in the provided context."""

def build_user_prompt(query, contexts):
    """Format the retrieved contexts + question into the user message."""
    context_block = "\n\n".join(
        f"[{i+1}] {c['document_text']}"
        for i, c in enumerate(contexts)
    )
    return f"Context:\n{context_block}\n\nQuestion: {query}\n\nAnswer:"

In [18]:
@torch.no_grad()
def generate(query: str, contexts: list, system_prompt: str = STRICT_PROMPT,
             max_new_tokens: int = 256) -> str:
    """Generate an answer given the query and retrieved contexts."""
    user_prompt = build_user_prompt(query, contexts)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Qwen uses a chat template — apply it
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm.device)

    output = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return answer

In [19]:
def run_query(query: str) -> dict:
    """
    Required interface for grading.
    Returns: {"answer": str, "retrieved_titles": List[str]}
    """
    try:
        if not query or not query.strip():
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": [],
            }

        candidates = retrieve(query, k=20)
        top_n = rerank(query, candidates, n=5)

        if not top_n:
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": [],
            }

        answer = generate(query, top_n, system_prompt=STRICT_PROMPT)

        return {
            "answer": answer,
            "retrieved_titles": [c['title'] for c in top_n],
        }

    except Exception as e:
        return {
            "answer": f"I don't have information about that in the provided context.",
            "retrieved_titles": [],
        }

In [20]:
test_inputs = [
    "Who directed Inception?",
    "asdfgh nonsense query",
    "",
    "Bir Zamanlar Anadolu'da hakkında ne biliyorsun?",
]
for q in test_inputs:
    r = run_query(q)
    assert isinstance(r, dict), f"must return dict, got {type(r)}"
    assert "answer" in r and "retrieved_titles" in r, f"missing keys: {list(r.keys())}"
    assert isinstance(r["answer"], str), f"answer must be str"
    assert isinstance(r["retrieved_titles"], list), f"retrieved_titles must be list"
    assert all(isinstance(t, str) for t in r["retrieved_titles"]), "all titles must be strings"
    print(f"OK: {q[:40]!r} -> {len(r['retrieved_titles'])} titles, ans={r['answer'][:80]!r}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


OK: 'Who directed Inception?' -> 5 titles, ans='Christopher Nolan directed Inception (2010). [Inception (2010)]'
OK: 'asdfgh nonsense query' -> 5 titles, ans="I don't have information about that in the provided context."
OK: '' -> 0 titles, ans="I don't have information about that in the provided context."
OK: "Bir Zamanlar Anadolu'da hakkında ne bili" -> 5 titles, ans="I don't have information about that in the provided context."


In [21]:
TASK4_QUERIES = [
    "Who directed Inception?",
    "What is the plot of The Godfather?",
    "Who directed Toy Story?",
    "Who directed Oppenheimer?",
    "A movie about a man who lives life backwards",
]

for q in TASK4_QUERIES:
    print(f"\n{'='*80}")
    print(f"Q: {q}")
    print('='*80)
    result = run_query(q)
    print(f"Retrieved titles: {result['retrieved_titles']}")
    print(f"\nAnswer: {result['answer']}")


Q: Who directed Inception?
Retrieved titles: ['Inception', 'Transcendence', 'Ambition', 'TerrorVision', 'Hollywood between Paranoia and Sci-Fi. The Power of Myth']

Answer: Christopher Nolan directed Inception (2010). [Inception (2010)]

Q: What is the plot of The Godfather?
Retrieved titles: ['The Godfather', 'The Godfather: Part III', 'The Godfather: Part II', 'The New Godfathers', 'Godfather']

Answer: The Godfather (1972). Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge. [The Godfather (1972)]

Q: Who directed Toy Story?
Retrieved titles: ['Toy Story', 'Toy Story 2', 'Toy Story 3', 'Toy Story of Terror!', 'Toy Story That Time Forgot']

Answer: John Lasseter directed Toy Story (1995). [Toy Story (1995)]

Q: Who directed Oppenhe